<a href="https://colab.research.google.com/github/cafekorea2000-prog/-healthcare/blob/main/%EB%B3%80%EB%B3%84%ED%95%B5%EC%8B%AC%EC%96%B4%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ─────────────────────────────────────────────────────────────────────
# 셀 K1: API 키 + 패키지 설치
# ─────────────────────────────────────────────────────────────────────
import getpass, os
api_key = getpass.getpass('ANTHROPIC_API_KEY 입력 후 엔터: ')
os.environ['ANTHROPIC_API_KEY'] = api_key
print('✅ 키 입력 완료' if api_key.startswith('sk-ant-') else '⚠️ 키 형식 확인 필요')

get_ipython().system('pip install anthropic pandas openpyxl --quiet')
print('✅ 패키지 설치 완료')

ANTHROPIC_API_KEY 입력 후 엔터: ··········
✅ 키 입력 완료
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.6/699.6 kB 9.5 MB/s eta 0:00:00
✅ 패키지 설치 완료


In [2]:
# ─────────────────────────────────────────────────────────────────────
# 셀 K2: stimuli_1674_최종.xlsx 업로드
# ─────────────────────────────────────────────────────────────────────
from google.colab import files
print('📂 "stimuli_1674_최종.xlsx" 파일을 선택하세요')
uploaded = files.upload()
INPUT_FILE = None
for fname in uploaded.keys():
    if fname.endswith('.xlsx'):
        INPUT_FILE = '/content/' + fname
        print(f'\n✅ 업로드 완료: {fname}')
        break

📂 "stimuli_1674_최종.xlsx" 파일을 선택하세요


Saving stimuli_1674_최종.xlsx to stimuli_1674_최종.xlsx

✅ 업로드 완료: stimuli_1674_최종.xlsx


In [3]:
# ─────────────────────────────────────────────────────────────────────
# 셀 K3: 변별 핵심어 추출 함수 정의 (출력 거의 없음)
# ─────────────────────────────────────────────────────────────────────
import json, time, re
import pandas as pd
from anthropic import Anthropic
from pathlib import Path
import concurrent.futures

client = Anthropic()
MODEL = 'claude-sonnet-4-6'
OUTPUT_DIR = Path('/content/keyterm_output')
OUTPUT_DIR.mkdir(exist_ok=True)
RESULT_FILE = OUTPUT_DIR / 'keyterms.jsonl'


SYSTEM_PROMPT = '''당신은 LLM 평가 자극 채점 보조 도구입니다. 자극 본문과 정답 표제어를 입력으로 받아, 응답이 정답으로 인정되려면 *반드시 포함되어야 할* 변별 핵심어를 출력합니다.

## 변별 핵심어 결정 규칙 (PD 지정)

**규칙 1**: 자극 본문이 인물명을 묻는 명시적 단서를 제공하는 경우 → 변별 핵심어는 표제어 중 *인물명* 부분만
- 예시 단서: "OOO의 이름을 딴", "OOO이 창시한", "OOO이 전수받은", "OOO의 제(制)"
- 예: 표제어 "서공철류 가야금산조" + 자극 "...인물(1911~1982)의 이름을 딴 가야금산조의 명칭..." → 변별핵심어: "서공철"

**규칙 2**: 자극 본문이 지역명을 묻는 명시적 단서를 제공하는 경우 → 변별 핵심어는 표제어 중 *지역명* 부분만
- 예시 단서: "어느 지역의", "OOO 지역에서 비롯된"
- 단, 자극이 "OO 지역에서 전승되는 무엇"이라고 *작품/장르명*을 묻고 있으면 → 표제어 전체

**규칙 3**: 그 외의 모든 경우 → 변별 핵심어는 *표제어 전체*
- 자극이 작품/곡/문헌/악기/용어의 *명칭 전체*를 묻고 있으면 표제어 그대로
- 예: 표제어 "통영개타령" + 자극 "경상도 향토민요 〈개타령〉을 통속화하여 만든 민요로... 이 곡의 명칭..." → 변별핵심어: "통영개타령"
- 예: 표제어 "학춤" + 자극 "학의 몸짓을 모방한 경남 지역 민속춤의 명칭..." → 변별핵심어: "학춤"
- 예: 표제어 "거문고" + 자극 "...한반도 고유의 현악기는 무엇입니까?" → 변별핵심어: "거문고"
- 예: 표제어 "황계사" + 자극 "...12가사 곡명은 무엇입니까?" → 변별핵심어: "황계사"

## 출력 형식 (JSON만, 다른 텍스트 X)

{"변별핵심어": "...", "이유": "한 줄로 짧게"}'''


def build_user_prompt(stim_text, target):
    return f'''[자극 본문]
{stim_text}

[정답 표제어]
{target}

위 자극에서 응답이 정답으로 인정되려면 반드시 포함되어야 할 변별 핵심어는 무엇입니까? JSON으로만 답하세요.'''


def extract_keyterm(stim_text, target, retry=0):
    """LLM으로 변별 핵심어 추출"""
    user_prompt = build_user_prompt(stim_text, target)
    try:
        response = client.messages.create(
            model=MODEL, max_tokens=300, temperature=0.0,
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': user_prompt}],
        )
        text = response.content[0].text.strip()
        if text.startswith('```'):
            lines = text.split('\n')
            text = '\n'.join(lines[1:-1] if lines[-1].startswith('```') else lines[1:])
        if text.startswith('json'):
            text = text[4:].strip()
        output = json.loads(text)
        keyterm = output.get('변별핵심어', '').strip()
        reason = output.get('이유', '').strip()
        usage = {
            'input': response.usage.input_tokens,
            'output': response.usage.output_tokens,
        }
        # 검증: 변별핵심어가 표제어와 무관한 단어가 나오면 표제어로 fallback
        if not keyterm or (keyterm not in target and target not in keyterm):
            keyterm = target
            reason = f'(fallback) {reason}'
        return keyterm, reason, usage, None
    except Exception as e:
        if retry < 2:
            time.sleep(2 * (retry + 1))
            return extract_keyterm(stim_text, target, retry + 1)
        return None, None, None, str(e)


print('✅ 함수 정의 완료')

✅ 함수 정의 완료


In [4]:
# ─────────────────────────────────────────────────────────────────────
# 셀 K4: 1,674 × 2 = 3,348개 자극 × 변별 핵심어 추출 (30~40분, ~$3)
# ─────────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE)
print(f'📂 자극 로드: {len(df)}개 표제어')

# 작업 목록 생성
tasks = []
for _, row in df.iterrows():
    tasks.append({
        'stim_id': f'{row["표제어"]}__multi',
        '표제어': row['표제어'],
        '카테고리': row['카테고리'],
        '조건': '다차원',
        '자극': row['다차원_자극'],
    })
    tasks.append({
        'stim_id': f'{row["표제어"]}__single',
        '표제어': row['표제어'],
        '카테고리': row['카테고리'],
        '조건': '단일차원',
        '자극': row['단일차원_자극'],
    })
print(f'   추출 대상: {len(tasks)}개 자극')

# 결과 파일 초기화
if RESULT_FILE.exists():
    RESULT_FILE.unlink()

results = []
fails = []
total_in, total_out = 0, 0


def process_task(task):
    keyterm, reason, usage, err = extract_keyterm(task['자극'], task['표제어'])
    return {
        'stim_id': task['stim_id'],
        '표제어': task['표제어'],
        '카테고리': task['카테고리'],
        '조건': task['조건'],
        '변별핵심어': keyterm,
        '이유': reason,
        '_usage': usage,
        '_error': err,
    }


print(f'\n{"="*60}')
print(f'🔄 변별 핵심어 추출 시작 (max_workers=10)')
print(f'{"="*60}\n')

start_time = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(process_task, t): t for t in tasks}
    completed = 0
    for future in concurrent.futures.as_completed(futures):
        completed += 1
        result = future.result()
        if result['변별핵심어'] is not None:
            results.append(result)
            if result['_usage']:
                total_in += result['_usage']['input']
                total_out += result['_usage']['output']
            with open(RESULT_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps(result, ensure_ascii=False) + '\n')
        else:
            fails.append(result)

        if completed % 200 == 0:
            elapsed = time.time() - start_time
            print(f'[{completed}/{len(tasks)}] {elapsed:.0f}s — 성공: {len(results)}, 실패: {len(fails)}')

elapsed = time.time() - start_time
cost = (total_in / 1_000_000) * 3.0 + (total_out / 1_000_000) * 15.0

print(f'\n{"="*60}')
print(f'✅ 추출 완료 — {elapsed/60:.1f}분, 비용 ${cost:.2f}')
print(f'{"="*60}')
print(f'   성공: {len(results)}개 / 실패: {len(fails)}개')

if fails:
    print(f'\n❌ 실패 사례:')
    for f in fails[:5]:
        print(f"  - {f['표제어']} ({f['조건']}): {f['_error'][:60] if f['_error'] else '?'}")

📂 자극 로드: 1674개 표제어
   추출 대상: 3348개 자극

🔄 변별 핵심어 추출 시작 (max_workers=10)

[200/3348] 33s — 성공: 200, 실패: 0
[400/3348] 65s — 성공: 400, 실패: 0
[600/3348] 99s — 성공: 600, 실패: 0
[800/3348] 131s — 성공: 800, 실패: 0
[1000/3348] 164s — 성공: 1000, 실패: 0
[1200/3348] 197s — 성공: 1200, 실패: 0
[1400/3348] 229s — 성공: 1400, 실패: 0
[1600/3348] 267s — 성공: 1599, 실패: 1
[1800/3348] 392s — 성공: 1599, 실패: 201
[2000/3348] 518s — 성공: 1599, 실패: 401


KeyboardInterrupt: 

In [5]:
# ─────────────────────────────────────────────────────────────────────
# 셀 K4-재처리: 누락 자극만 안전하게 재처리
# ─────────────────────────────────────────────────────────────────────
import json

# 1. 이미 처리된 자극 확인
done_ids = set()
with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            done_ids.add(json.loads(line)['stim_id'])
        except:
            pass

print(f'이미 처리됨: {len(done_ids)}개')

# 2. 누락 자극 추출
df = pd.read_excel(INPUT_FILE)
remaining_tasks = []
for _, row in df.iterrows():
    multi_id = f'{row["표제어"]}__multi'
    single_id = f'{row["표제어"]}__single'
    if multi_id not in done_ids:
        remaining_tasks.append({
            'stim_id': multi_id,
            '표제어': row['표제어'],
            '카테고리': row['카테고리'],
            '조건': '다차원',
            '자극': row['다차원_자극'],
        })
    if single_id not in done_ids:
        remaining_tasks.append({
            'stim_id': single_id,
            '표제어': row['표제어'],
            '카테고리': row['카테고리'],
            '조건': '단일차원',
            '자극': row['단일차원_자극'],
        })

print(f'재처리 대상: {len(remaining_tasks)}개')

if not remaining_tasks:
    print('✅ 모든 자극 처리 완료. 셀 K5로 진행하세요.')
else:
    # 3. 안전한 재처리 (max_workers=3 + 호출 간 sleep)
    new_results = []
    new_fails = []
    new_in, new_out = 0, 0

    print(f'\n{"="*60}')
    print(f'🔄 누락 자극 재처리 (max_workers=3, 안전 모드)')
    print(f'{"="*60}\n')

    start_time = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = {executor.submit(process_task, t): t for t in remaining_tasks}
        completed = 0
        for future in concurrent.futures.as_completed(futures):
            completed += 1
            result = future.result()
            if result['변별핵심어'] is not None:
                new_results.append(result)
                if result['_usage']:
                    new_in += result['_usage']['input']
                    new_out += result['_usage']['output']
                with open(RESULT_FILE, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(result, ensure_ascii=False) + '\n')
                # 누적 results에도 추가
                results.append(result)
            else:
                new_fails.append(result)

            if completed % 50 == 0:
                elapsed = time.time() - start_time
                print(f'[{completed}/{len(remaining_tasks)}] {elapsed:.0f}s — 성공: {len(new_results)}, 실패: {len(new_fails)}')

    elapsed = time.time() - start_time
    cost = (new_in / 1_000_000) * 3.0 + (new_out / 1_000_000) * 15.0

    print(f'\n{"="*60}')
    print(f'✅ 재처리 완료 — {elapsed/60:.1f}분, 추가 비용 ${cost:.2f}')
    print(f'{"="*60}')
    print(f'   재처리 성공: {len(new_results)}개')
    print(f'   재처리 실패: {len(new_fails)}개')
    print(f'   전체 누적 성공: {len(results)}개 / 3,348개')

    if new_fails:
        print(f'\n❌ 재처리에서도 실패한 사례 (앞 5개):')
        for f in new_fails[:5]:
            print(f"  - {f['표제어']} ({f['조건']}): {f['_error'][:60] if f['_error'] else '?'}")

이미 처리됨: 1591개
재처리 대상: 1749개

🔄 누락 자극 재처리 (max_workers=3, 안전 모드)

[50/1749] 26s — 성공: 50, 실패: 0
[100/1749] 52s — 성공: 100, 실패: 0
[150/1749] 79s — 성공: 150, 실패: 0
[200/1749] 108s — 성공: 200, 실패: 0
[250/1749] 134s — 성공: 250, 실패: 0
[300/1749] 162s — 성공: 300, 실패: 0
[350/1749] 189s — 성공: 350, 실패: 0
[400/1749] 216s — 성공: 400, 실패: 0
[450/1749] 242s — 성공: 450, 실패: 0
[500/1749] 268s — 성공: 500, 실패: 0
[550/1749] 294s — 성공: 550, 실패: 0
[600/1749] 323s — 성공: 600, 실패: 0
[650/1749] 351s — 성공: 650, 실패: 0
[700/1749] 379s — 성공: 700, 실패: 0
[750/1749] 407s — 성공: 750, 실패: 0
[800/1749] 435s — 성공: 800, 실패: 0
[850/1749] 462s — 성공: 850, 실패: 0
[900/1749] 489s — 성공: 900, 실패: 0
[950/1749] 515s — 성공: 950, 실패: 0
[1000/1749] 544s — 성공: 1000, 실패: 0
[1050/1749] 571s — 성공: 1050, 실패: 0
[1100/1749] 599s — 성공: 1100, 실패: 0
[1150/1749] 625s — 성공: 1150, 실패: 0
[1200/1749] 652s — 성공: 1200, 실패: 0
[1250/1749] 680s — 성공: 1250, 실패: 0
[1300/1749] 708s — 성공: 1300, 실패: 0
[1350/1749] 736s — 성공: 1350, 실패: 0
[1400/1749] 765s — 성공: 1400, 실패: 

In [6]:
# ─────────────────────────────────────────────────────────────────────
# 셀 K5: 결과 검증 + 다운로드
# ─────────────────────────────────────────────────────────────────────
from google.colab import files

# 결과 파일에서 다시 로드 (재처리 후 results 변수가 갱신되었지만 안전을 위해 파일에서 재로드)
all_results = []
with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        all_results.append(json.loads(line))

print(f'결과 파일 로드: {len(all_results)}개')

# DataFrame 생성
df_keyterms = pd.DataFrame([{
    '표제어': r['표제어'],
    '카테고리': r['카테고리'],
    '조건': r['조건'],
    '변별핵심어': r['변별핵심어'],
    '이유': r['이유'],
} for r in all_results])

# 다차원/단일차원 분리해서 자극 파일에 컬럼 추가
df_multi = df_keyterms[df_keyterms['조건'] == '다차원'].rename(
    columns={'변별핵심어': '다차원_변별핵심어', '이유': '다차원_이유'}
).drop(columns='조건')
df_single = df_keyterms[df_keyterms['조건'] == '단일차원'].rename(
    columns={'변별핵심어': '단일차원_변별핵심어', '이유': '단일차원_이유'}
).drop(columns='조건')

# 원본 자극 파일에 변별 핵심어 컬럼 추가
df_orig = pd.read_excel(INPUT_FILE)
df_merged = df_orig.merge(
    df_multi[['표제어', '카테고리', '다차원_변별핵심어', '다차원_이유']],
    on=['표제어', '카테고리'], how='left'
).merge(
    df_single[['표제어', '카테고리', '단일차원_변별핵심어', '단일차원_이유']],
    on=['표제어', '카테고리'], how='left'
)

OUTPUT_EXCEL = '/content/keyterm_output/stimuli_1674_with_keyterms.xlsx'
df_merged.to_excel(OUTPUT_EXCEL, index=False)
files.download(OUTPUT_EXCEL)
files.download(str(RESULT_FILE))

print(f'\n✅ 다운로드 완료: stimuli_1674_with_keyterms.xlsx ({len(df_merged)}개 자극, 변별 핵심어 포함)')

# ── 검증 통계 ──
print(f'\n=== 변별 핵심어 분포 ===')
print(f'\n[다차원 조건]')
multi_full = (df_merged['다차원_변별핵심어'] == df_merged['표제어']).sum()
multi_partial = len(df_merged) - multi_full
print(f'  표제어 전체가 변별핵심어: {multi_full}개 ({multi_full/len(df_merged)*100:.1f}%)')
print(f'  표제어 일부(인물/지역)가 변별핵심어: {multi_partial}개 ({multi_partial/len(df_merged)*100:.1f}%)')

print(f'\n[단일차원 조건]')
single_full = (df_merged['단일차원_변별핵심어'] == df_merged['표제어']).sum()
single_partial = len(df_merged) - single_full
print(f'  표제어 전체가 변별핵심어: {single_full}개 ({single_full/len(df_merged)*100:.1f}%)')
print(f'  표제어 일부(인물/지역)가 변별핵심어: {single_partial}개 ({single_partial/len(df_merged)*100:.1f}%)')

print(f'\n=== 변별핵심어가 표제어 일부인 사례 (다차원 조건, 첫 15개) ===')
partial_examples = df_merged[df_merged['다차원_변별핵심어'] != df_merged['표제어']].head(15)
for _, r in partial_examples.iterrows():
    reason = str(r['다차원_이유'])[:50] if pd.notna(r['다차원_이유']) else ''
    print(f"  {r['표제어']:<25} → '{r['다차원_변별핵심어']}' ({reason})")

print(f'\n=== 변별핵심어가 표제어 일부인 사례 (단일차원 조건, 첫 15개) ===')
partial_examples = df_merged[df_merged['단일차원_변별핵심어'] != df_merged['표제어']].head(15)
for _, r in partial_examples.iterrows():
    reason = str(r['단일차원_이유'])[:50] if pd.notna(r['단일차원_이유']) else ''
    print(f"  {r['표제어']:<25} → '{r['단일차원_변별핵심어']}' ({reason})")

print(f'\n💡 다음 단계:')
print(f'   1. 다운받은 stimuli_1674_with_keyterms.xlsx를 직접 열어 검증')
print(f'   2. 변별핵심어 컬럼이 합리적인지 50~100개 샘플 검토')
print(f'   3. 잘못된 추출이 발견되면 알려주세요 (수동 보정 가능)')

결과 파일 로드: 3348개


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ 다운로드 완료: stimuli_1674_with_keyterms.xlsx (1692개 자극, 변별 핵심어 포함)

=== 변별 핵심어 분포 ===

[다차원 조건]
  표제어 전체가 변별핵심어: 1617개 (95.6%)
  표제어 일부(인물/지역)가 변별핵심어: 75개 (4.4%)

[단일차원 조건]
  표제어 전체가 변별핵심어: 1597개 (94.4%)
  표제어 일부(인물/지역)가 변별핵심어: 95개 (5.6%)

=== 변별핵심어가 표제어 일부인 사례 (다차원 조건, 첫 15개) ===
  가척(歌尺)                    → '가척' (자극이 용어의 명칭 전체를 묻고 있으므로 표제어 전체가 변별핵심어)
  가척(笳尺)                    → '가척' (자극이 용어 명칭 전체를 묻고 있으므로 표제어 전체가 변별핵심어)
  강태홍류 가야금산조                → '강태홍' (자극이 '강태홍(1893~1957)이 완성한'이라는 인물명 단서를 명시적으로 제공하며 산조)
  길군악(가사)                   → '길군악' (자극이 성악곡의 명칭 전체를 묻고 있으나, 표제어의 괄호 부분(가사)은 장르 분류 표기이므)
  길군악(취타풍류)                 → '길군악' (자극이 악곡의 명칭 전체를 묻고 있으며, 표제어에서 괄호 안 '취타풍류'는 출처 표기이므로)
  김병호류 가야금산조                → '김병호' (자극이 창시자 인물명(김병호)을 명시적으로 제공하며 해당 인물의 이름을 딴 유파명을 묻고 )
  김일구류 아쟁산조                 → '김일구' (자극이 특정 인물의 이름을 딴 아쟁산조 유파명을 묻고 있으므로 인물명 부분만 변별핵심어로 )
  놀량(경기)                    → '놀량' (자극이 작품/곡의 명칭 전체를 묻고 있으며, 표제어에서 지역명(경기)은 괄호 안 부가 정보)
  대국 1·2·3                  → '대국' (자극이